<a href="https://colab.research.google.com/github/ailtoncnascimento/ailtoncnascimento.github.io/blob/main/Nonlinerities_Explicitation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
step1_Ecoeffs.py
----------------
Verification of the expansion coefficients E_{r,k} defined by

        (d_x)^r = sum_k E_{r,k} (d_y)^k ,        d_x = s d_y ,

which are used in the two inverse (spatial) formulas of the pointwise section.

The script computes (s d_y)^r symbolically for r = 2,...,9, extracts the
coefficient of (d_y)^k for k = r, r-1, r-2, r-3, and compares it with the
closed formulas

        E_{r,r}   = s^r,
        E_{r,r-1} = C(r,2) s^{r-2} s_x,
        E_{r,r-2} = C(r,3) s^{r-3} s_xx + 3 C(r,4) s^{r-4} s_x^2,
        E_{r,r-3} = C(r,4) s^{r-4} s_xxx + 10 C(r,5) s^{r-5} s_x s_xx
                    + 15 C(r,6) s^{r-6} s_x^3,

where s_x = d_x s = s s_y, s_xx = d_x s_x, s_xxx = d_x s_xx.
"""

import sympy as sp

y = sp.Symbol('y')
s = sp.Function('s', positive=True)(y)
f = sp.Function('f')(y)
C = sp.binomial

# x-derivatives of s written in the y variable (d_x = s d_y)
sx = sp.expand(s*sp.diff(s, y))
sxx = sp.expand(s*sp.diff(sx, y))
sxxx = sp.expand(s*sp.diff(sxx, y))


def op_power(r):
    """(s d_y)^r applied to a generic function f."""
    e = f
    for _ in range(r):
        e = sp.expand(s*sp.diff(e, y))
    return sp.expand(e)


def E_formula(r, k):
    d = r - k
    if d == 0:
        return s**r
    if d == 1:
        return C(r, 2)*s**(r-2)*sx
    if d == 2:
        return C(r, 3)*s**(r-3)*sxx + 3*C(r, 4)*s**(r-4)*sx**2
    if d == 3:
        return (C(r, 4)*s**(r-4)*sxxx + 10*C(r, 5)*s**(r-5)*sx*sxx
                + 15*C(r, 6)*s**(r-6)*sx**3)
    raise ValueError(d)


def main():
    ok = True
    for r in range(2, 10):
        e = op_power(r)
        for k in range(r, max(r-4, -1), -1):
            coeff = e.coeff(sp.Derivative(f, (y, k))) if k > 0 else e.coeff(f)
            if sp.simplify(sp.expand(coeff - E_formula(r, k))) != 0:
                ok = False
                print("  MISMATCH  r=%d  k=%d" % (r, k))
    print("E_{r,k} formulas verified for 2 <= r <= 9, r-3 <= k <= r :", ok)
    return ok


if __name__ == "__main__":
    main()

E_{r,k} formulas verified for 2 <= r <= 9, r-3 <= k <= r : True


In [2]:
"""
step2_chain.py
--------------
End-to-end verification of the reduction chain of the pointwise section for the odd
orders n = 5, 7, 9, 11, 13, 15.

Starting from the prescribed unreduced coefficients

        a^{(0)}_n     = eps A,
        a^{(0)}_{n-1} = eps (n+1)/2 A_x,
        a^{(0)}_{n-2} = B_{n-2},
        a^{(0)}_{n-3} = B_{n-3},

with B_{n-2}, B_{n-3} given by the two boxed reversal formulas, and with
Lambda_{n-2}, Lambda_{n-3} treated as free symbols, the script applies

        (0) -> (1)  spatial diffeomorphism      (coefficients E_{r,k}),
        (1) -> (2)  time normalisation          (factor mbar/m),
        (2) -> (3)  multiplier q = A^{-1/n}     (Leibniz/gauge formula),

and checks that

        a^{(2)}_n     = eps mbar,
        a^{(2)}_{n-1} = eps mbar p/s,
        a^{(3)}_{n-1} = 0,
        a^{(3)}_{n-2} = mbar m^{-2/n} Lambda_{n-2}   (independent of x),
        a^{(3)}_{n-3} = mbar m^{-3/n} Lambda_{n-3}   (independent of x),

as well as the intermediate identities (a2-n2) and (a2-n3) for a^{(2)}_{n-2}
and a^{(2)}_{n-3}.  The background A = A(x) is a generic positive function;
m and mbar are free positive symbols, so the algebra tested is exactly the
pointwise algebra of the section.

The script also prints the two constants entering the fifth-order
obstruction discussed in the sharpness remark.
"""

import sympy as sp

x = sp.Symbol('x', real=True)
m, mb = sp.symbols('m mbar', positive=True)
L2, L3 = sp.symbols('Lambda2 Lambda3')
A = sp.Function('A', positive=True)(x)
C = sp.binomial


def run(n, verbose=True):
    j = (n - 1)//2
    eps = (-1)**(j + 1)
    R = sp.Rational
    s = (m/A)**R(1, n)
    q = A**R(-1, n)
    p = sp.diff(A, x)/A
    Ax, Axx, Axxx = sp.diff(A, x), sp.diff(A, x, 2), sp.diff(A, x, 3)
    sx, sxx, sxxx = sp.diff(s, x), sp.diff(s, x, 2), sp.diff(s, x, 3)

    def Dy(g):
        return sp.diff(g, x)/s

    qy = Dy(q)
    qyy = Dy(qy)
    qyyy = Dy(qyy)

    def E(r, k):
        d = r - k
        if d == 0:
            return s**r
        if d == 1:
            return C(r, 2)*s**(r-2)*sx
        if d == 2:
            return C(r, 3)*s**(r-3)*sxx + 3*C(r, 4)*s**(r-4)*sx**2
        if d == 3:
            return (C(r, 4)*s**(r-4)*sxxx + 10*C(r, 5)*s**(r-5)*sx*sxx
                    + 15*C(r, 6)*s**(r-6)*sx**3)
        raise ValueError(d)

    # ---- differential identities for s and q -----------------------------
    lemma = {
        "s_x/s":   sx/s - (-p/n),
        "s_xx/s":  sxx/s - (-sp.diff(p, x)/n + p**2/n**2),
        "s_xxx/s": sxxx/s - (-sp.diff(p, x, 2)/n + 3*p*sp.diff(p, x)/n**2
                             - p**3/n**3),
        "q_y/q":   qy/q - (-p/(n*s)),
        "q_yy/q":  qyy/q - (-sp.diff(p, x)/(n*s**2)),
        "q_yyy/q": qyyy/q - (-sp.diff(p, x, 2)/(n*s**3)
                             - p*sp.diff(p, x)/(n**2*s**3)),
    }
    lemma_ok = all(sp.simplify(v) == 0 for v in lemma.values())

    # ---- prescribed unreduced coefficients -------------------------------
    B2 = L2*A**R(n-2, n) + eps*R(n**2-1, 6)*(Axx - R(n-2, 4*n)*Ax**2/A)
    B3 = (L3*A**R(n-3, n) + R((n-1)*(n-2), 2*n)*L2*A**R(-2, n)*Ax
          + eps*R((n-2)*(n**2-1), 24)*(Axxx - R(n-1, n)*Ax*Axx/A
                                       + R(n-1, 2*n)*Ax**3/A**2))
    a0 = {n: eps*A, n-1: eps*R(n+1, 2)*Ax, n-2: B2, n-3: B3}

    # ---- the three transformations ---------------------------------------
    a1 = {k: sum(a0[r]*E(r, k) for r in range(k, n+1) if r in a0)
          for k in (n, n-1, n-2, n-3)}
    a2 = {k: (mb/m)*a1[k] for k in a1}
    a3 = {}
    a3[n-1] = a2[n-1] + n*a2[n]*qy/q
    a3[n-2] = a2[n-2] + (n-1)*a2[n-1]*qy/q + C(n, 2)*a2[n]*qyy/q
    a3[n-3] = (a2[n-3] + (n-2)*a2[n-2]*qy/q + C(n-1, 2)*a2[n-1]*qyy/q
               + C(n, 3)*a2[n]*qyyy/q)

    b2 = mb*m**R(-2, n)*L2
    b3 = mb*m**R(-3, n)*L3
    tests = {
        "conjugator identities":  0 if lemma_ok else 1,
        "a2_n":             a2[n] - eps*mb,
        "a2_{n-1}":         a2[n-1] - eps*mb*p/s,
        "a3_{n-1} = 0":     a3[n-1],
        "a3_{n-2}":         a3[n-2] - b2,
        "a3_{n-3}":         a3[n-3] - b3,
        "(a2-n2)":           a2[n-2] - (b2 + eps*mb*(n-1)/s**2
                                       * (sp.diff(p, x)/2 + p**2/n)),
        "(a2-n3)":           a2[n-3] - (b3 + R(n-2, n)*p/s*b2
                                       + eps*mb*(n-1)*(n-2)/s**3
                                       * (sp.diff(p, x, 2)/6
                                          + R(7, 6*n)*p*sp.diff(p, x)
                                          + p**3/n**2)),
    }
    res = {k: sp.simplify(v) == 0 for k, v in tests.items()}
    if verbose:
        print("n=%2d  eps=%+d : " % (n, eps)
              + "  ".join("%s:%s" % (k, "OK" if v else "FAIL")
                          for k, v in res.items()))
    return all(res.values())


def main():
    ok = all(run(n) for n in (5, 7, 9, 11, 13, 15))
    print("all orders verified:", ok)
    print()
    print("fifth-order obstruction constants:")
    for n in (5, 7, 9):
        print("   n=%2d :  d_{z2} B_{n-2} = (n^2-1)/6 = %s ,"
              "   d_{z3} B_{n-3} = (n-2)(n^2-1)/24 = %s"
              % (n, sp.Rational(n**2-1, 6), sp.Rational((n-2)*(n**2-1), 24)))
    print("   at n=5 the two jet slots coincide, so the mixed-jet identity")
    print("   would require 4 = 3.")
    return ok


if __name__ == "__main__":
    main()

n= 5  eps=-1 : conjugator identities:OK  a2_n:OK  a2_{n-1}:OK  a3_{n-1} = 0:OK  a3_{n-2}:OK  a3_{n-3}:OK  (a2-n2):OK  (a2-n3):OK
n= 7  eps=+1 : conjugator identities:OK  a2_n:OK  a2_{n-1}:OK  a3_{n-1} = 0:OK  a3_{n-2}:OK  a3_{n-3}:OK  (a2-n2):OK  (a2-n3):OK
n= 9  eps=-1 : conjugator identities:OK  a2_n:OK  a2_{n-1}:OK  a3_{n-1} = 0:OK  a3_{n-2}:OK  a3_{n-3}:OK  (a2-n2):OK  (a2-n3):OK
n=11  eps=+1 : conjugator identities:OK  a2_n:OK  a2_{n-1}:OK  a3_{n-1} = 0:OK  a3_{n-2}:OK  a3_{n-3}:OK  (a2-n2):OK  (a2-n3):OK
n=13  eps=-1 : conjugator identities:OK  a2_n:OK  a2_{n-1}:OK  a3_{n-1} = 0:OK  a3_{n-2}:OK  a3_{n-3}:OK  (a2-n2):OK  (a2-n3):OK
n=15  eps=+1 : conjugator identities:OK  a2_n:OK  a2_{n-1}:OK  a3_{n-1} = 0:OK  a3_{n-2}:OK  a3_{n-3}:OK  (a2-n2):OK  (a2-n3):OK
all orders verified: True

fifth-order obstruction constants:
   n= 5 :  d_{z2} B_{n-2} = (n^2-1)/6 = 4 ,   d_{z3} B_{n-3} = (n-2)(n^2-1)/24 = 3
   n= 7 :  d_{z2} B_{n-2} = (n^2-1)/6 = 8 ,   d_{z3} B_{n-3} = (n-2)(n^2-1)/24 = 

In [3]:
"""
step3_symbolic.py
-----------------
The same end-to-end verification as step2_chain.py, but with the order n
kept as a symbol.  The sign eps is carried as a symbol e subject to the
relation e^2 = 1, which is imposed by the helper kill_e.

This removes any doubt that the identities of the pointwise section are order-by-order
coincidences: they hold as identities in n.
"""

import sympy as sp

x = sp.Symbol('x', real=True)
m, mb = sp.symbols('m mbar', positive=True)
L2, L3 = sp.symbols('Lambda2 Lambda3')
n = sp.Symbol('n', positive=True, integer=True)
e = sp.Symbol('e')                      # eps, with e**2 = 1
A = sp.Function('A', positive=True)(x)
C = sp.binomial


def kill_e(expr):
    """Impose eps^2 = 1 and simplify."""
    expr = sp.expand(expr)
    return sp.simplify(expr.replace(lambda a: a.is_Pow and a.base == e,
                                    lambda a: e**(a.exp % 2)))


def main():
    s = (m/A)**(1/n)
    q = A**(-1/n)
    p = sp.diff(A, x)/A
    Ax, Axx, Axxx = sp.diff(A, x), sp.diff(A, x, 2), sp.diff(A, x, 3)
    sx, sxx, sxxx = sp.diff(s, x), sp.diff(s, x, 2), sp.diff(s, x, 3)

    def Dy(g):
        return sp.diff(g, x)/s

    qy = Dy(q)
    qyy = Dy(qy)
    qyyy = Dy(qyy)

    def E(r, k):
        d = r - k
        if d == 0:
            return s**r
        if d == 1:
            return C(r, 2)*s**(r-2)*sx
        if d == 2:
            return C(r, 3)*s**(r-3)*sxx + 3*C(r, 4)*s**(r-4)*sx**2
        if d == 3:
            return (C(r, 4)*s**(r-4)*sxxx + 10*C(r, 5)*s**(r-5)*sx*sxx
                    + 15*C(r, 6)*s**(r-6)*sx**3)
        raise ValueError(d)

    B2 = L2*A**((n-2)/n) + e*(n**2-1)/6*(Axx - (n-2)/(4*n)*Ax**2/A)
    B3 = (L3*A**((n-3)/n) + (n-1)*(n-2)/(2*n)*L2*A**(-sp.Integer(2)/n)*Ax
          + e*(n-2)*(n**2-1)/24*(Axxx - (n-1)/n*Ax*Axx/A
                                 + (n-1)/(2*n)*Ax**3/A**2))
    # keyed by codimension: 0 <-> n, 1 <-> n-1, 2 <-> n-2, 3 <-> n-3
    a0 = {0: e*A, 1: e*(n+1)/2*Ax, 2: B2, 3: B3}

    a1 = {k: sum(a0[r]*E(n-r, n-k) for r in range(k+1)) for k in range(4)}
    a2 = {k: (mb/m)*a1[k] for k in a1}
    a3 = {}
    a3[1] = a2[1] + n*a2[0]*qy/q
    a3[2] = a2[2] + (n-1)*a2[1]*qy/q + C(n, 2)*a2[0]*qyy/q
    a3[3] = (a2[3] + (n-2)*a2[2]*qy/q + C(n-1, 2)*a2[1]*qyy/q
             + C(n, 3)*a2[0]*qyyy/q)

    checks = {
        "a^{(2)}_n     = eps mbar":              a2[0] - e*mb,
        "a^{(2)}_{n-1} = eps mbar p/s":          a2[1] - e*mb*p/s,
        "a^{(3)}_{n-1} = 0":                     a3[1],
        "a^{(3)}_{n-2} = mbar m^{-2/n} Lam_2":   a3[2] - mb*m**(-sp.Integer(2)/n)*L2,
        "a^{(3)}_{n-3} = mbar m^{-3/n} Lam_3":   a3[3] - mb*m**(-sp.Integer(3)/n)*L3,
    }
    ok = True
    for k, v in checks.items():
        good = kill_e(v) == 0
        ok = ok and good
        print("%-38s : %s" % (k, "OK" if good else "FAIL"))
    print("symbolic-n verification:", ok)
    return ok


if __name__ == "__main__":
    main()

a^{(2)}_n     = eps mbar               : OK
a^{(2)}_{n-1} = eps mbar p/s           : OK
a^{(3)}_{n-1} = 0                      : OK
a^{(3)}_{n-2} = mbar m^{-2/n} Lam_2    : OK
a^{(3)}_{n-3} = mbar m^{-3/n} Lam_3    : OK
symbolic-n verification: True


In [4]:
"""
step4_tables_examples.py
------------------------
Three independent checks.

(a)  The totals of the two coefficient tables in the proofs of
     the two reversal propositions, and the factorised form

        eps A (n-2)(n^2-1)/24 [ p_xx + (2n+1)/n p p_x + (n+1)/(2n) p^3 ].

(b)  The four highest jet derivatives of the quadratic block N_H, and its
     divergence representation

        N_H = d_x^{j+1}( u d_x^j u ) - (eps/2) d_x( (d_x^j u)^2 ),

     for n = 7, 9, ..., 17 (the case n = 7, in which z_j z_{j+1} also feeds
     the (n-3) slot, is included).

(c)  Every coefficient of the five worked examples of N_ann printed after
     the annihilating-core corollary: the quadratic coefficients

        binom(j+1,r) + [ (n^2-1)/24 if r=2 ] + [ (n-1)(n^2-1)/48 if r=3 ]

     and the three rational corrections.
"""

import sympy as sp

n = sp.Symbol('n', positive=True)
C = sp.binomial


def part_a():
    rows2 = {
        "reversed multiplier": ((n-1)/2, (n-1)/n),
        "-eps A E_{n,n-2}":    ((n-1)*(n-2)/6, -(n-1)*(n-2)*(3*n-5)/(24*n)),
        "order n-1 term":      (0, (n-1)*(n-2)*(n+1)/(4*n)),
    }
    tot2 = ((n**2-1)/6, (n**2-1)*(3*n+2)/(24*n))
    ok2 = all(sp.simplify(sum(r[i] for r in rows2.values()) - tot2[i]) == 0
              for i in range(2))

    rows3 = {
        "reversed multiplier": ((n-1)*(n-2)/6, 7*(n-1)*(n-2)/(6*n),
                                (n-1)*(n-2)/n**2),
        "-eps A E_{n,n-3}":    ((n-1)*(n-2)*(n-3)/24,
                                -(n-1)*(n-2)*(n-3)*(2*n-5)/(24*n),
                                (n-1)*(n-2)**2*(n-3)**2/(48*n**2)),
        "order n-1 term":      (0, (n-1)*(n-2)*(n-3)*(n+1)/(12*n),
                                -(n-1)*(n-2)*(n-3)*(n+1)*(3*n-8)/(48*n**2)),
        "order n-2 coupling":  (0, (n-1)*(n-2)*(n-3)*(n+1)/(12*n),
                                (n-1)*(n-2)*(n-3)*(n+1)*(3*n+2)/(48*n**2)),
    }
    tot3 = ((n-2)*(n**2-1)/24, (n-2)*(n**2-1)*(2*n+1)/(24*n),
            (n-2)*(n**2-1)*(n+1)/(48*n))
    ok3 = all(sp.simplify(sum(r[i] for r in rows3.values()) - tot3[i]) == 0
              for i in range(3))

    fac = (n-2)*(n**2-1)/24
    okf = all(sp.simplify(tot3[i]/fac - c) == 0 for i, c in
              enumerate([1, (2*n+1)/n, (n+1)/(2*n)]))
    print("(a) table totals: order n-2 :", ok2, " order n-3 :", ok3,
          " factorisation :", okf)
    return ok2 and ok3 and okf


def part_b():
    xx = sp.Symbol('x')
    u = sp.Function('u')(xx)
    ok = True
    for N in (7, 9, 11, 13, 15, 17):
        j = (N-1)//2
        eps = (-1)**(j+1)
        z = [sp.Symbol('z%d' % i) for i in range(N+1)]
        NH = (sum(sp.binomial(j+1, r)*z[r]*z[N-r] for r in range(j))
              + (j+2-eps)*z[j]*z[j+1])
        jets = [sp.simplify(sp.diff(NH, z[N]) - z[0]) == 0,
                sp.simplify(sp.diff(NH, z[N-1]) - sp.Rational(N+1, 2)*z[1]) == 0,
                sp.simplify(sp.diff(NH, z[N-2])
                            - sp.Rational(N**2-1, 8)*z[2]) == 0,
                sp.simplify(sp.diff(NH, z[N-3])
                            - sp.Rational((N**2-1)*(N-3), 48)*z[3]) == 0]
        dv = sp.expand(sp.diff(u*sp.diff(u, xx, j), xx, j+1)
                       - sp.Rational(eps, 2)*sp.diff(sp.diff(u, xx, j)**2, xx))
        sub = {sp.Derivative(u, (xx, i)): z[i] for i in range(1, N+1)}
        dv = sp.expand(dv.subs(sub).subs(u, z[0]))
        div = sp.simplify(dv - NH) == 0
        ok = ok and all(jets) and div
        print("(b) n=%2d : jet derivatives %s , divergence form %s"
              % (N, all(jets), div))
    return ok


def part_c():
    paper = {
        7:  ([(0, 1), (1, 4), (2, 8), (3, 10)],
             sp.Rational(-10, 7), sp.Rational(-60, 7), sp.Rational(30, 7)),
        9:  ([(0, 1), (1, 5), (2, sp.Rational(40, 3)), (3, sp.Rational(70, 3)),
              (4, 7)],
             sp.Rational(70, 27), sp.Rational(560, 27), sp.Rational(280, 27)),
        11: ([(0, 1), (1, 6), (2, 20), (3, 45), (4, 15), (5, 6)],
             sp.Rational(-45, 11), sp.Rational(-450, 11), sp.Rational(225, 11)),
        13: ([(0, 1), (1, 7), (2, 28), (3, 77), (4, 35), (5, 21), (6, 9)],
             sp.Rational(77, 13), sp.Rational(924, 13), sp.Rational(462, 13)),
        15: ([(0, 1), (1, 8), (2, sp.Rational(112, 3)), (3, sp.Rational(364, 3)),
              (4, 70), (5, 56), (6, 28), (7, 8)],
             sp.Rational(-364, 45), sp.Rational(-5096, 45),
             sp.Rational(2548, 45)),
    }
    ok = True
    for N, (quad, c1, c2, c3) in paper.items():
        j = (N-1)//2
        eps = (-1)**(j+1)
        base = {r: (sp.binomial(j+1, r) if r < j else (j+2-eps))
                for r in range(j+1)}
        corr = {2: sp.Rational(N**2-1, 24), 3: sp.Rational((N-1)*(N**2-1), 48)}
        good = (len(quad) == j+1
                and all(sp.simplify(base[r] + corr.get(r, 0) - c) == 0
                        for r, c in quad))
        g1 = sp.simplify(-eps*sp.Rational((N-2)*(N**2-1), 24*N) - c1) == 0
        g2 = sp.simplify(-eps*sp.Rational((N-1)*(N-2)*(N**2-1), 24*N) - c2) == 0
        g3 = sp.simplify(sp.Rational((N-1)*(N-2)*(N**2-1), 48*N) - c3) == 0
        ok = ok and good and g1 and g2 and g3
        print("(c) n=%2d : quadratic block %s , corrections %s %s %s"
              % (N, good, g1, g2, g3))
    return ok


def main():
    ok = part_a() and part_b() and part_c()
    print("all table, block and example checks:", ok)
    return ok


if __name__ == "__main__":
    main()

(a) table totals: order n-2 : True  order n-3 : True  factorisation : True
(b) n= 7 : jet derivatives True , divergence form True
(b) n= 9 : jet derivatives True , divergence form True
(b) n=11 : jet derivatives True , divergence form True
(b) n=13 : jet derivatives True , divergence form True
(b) n=15 : jet derivatives True , divergence form True
(b) n=17 : jet derivatives True , divergence form True
(c) n= 7 : quadratic block True , corrections True True True
(c) n= 9 : quadratic block True , corrections True True True
(c) n=11 : quadratic block True , corrections True True True
(c) n=13 : quadratic block True , corrections True True True
(c) n=15 : quadratic block True , corrections True True True
all table, block and example checks: True


In [5]:
"""
step5b_rows.py
--------------
Row-by-row verification of the two coefficient tables, i.e. of each single
entry rather than only of the totals checked in step4_tables_examples.py.

Method.  The background is realised as a jet at one point:

        A = exp(P),   P(x) = c1 x + c2 x^2/2 + c3 x^3/6,

so that at x = 0 one has A = 1, p = c1, p_x = c2, p_xx = c3, and every
contribution becomes a polynomial in (c1, c2, c3) whose coefficients can be
extracted unambiguously.

Remark on methodology.  A first attempt extracted the coefficients directly
from expressions containing the symbolic powers s^{-(n-2)} and (m/A)^{1/n}.
For a generic (not manifestly positive) background, sympy does not combine
those powers, and the extraction returned false negatives on the rows in
which the combination s^{-(n-2)} s^{-2} m = A is needed.  The jet
substitution used here avoids symbolic powers of a function altogether and
is the reliable variant; it reproduces exactly the entries printed in the
paper.
"""

import sympy as sp

x = sp.Symbol('x', real=True)
m, mb = sp.symbols('m mbar', positive=True)
n = sp.Symbol('n', positive=True)
c1, c2, c3 = sp.symbols('c1 c2 c3')          # p, p_x, p_xx at x = 0
P = c1*x + c2*x**2/2 + c3*x**3/6
A = sp.exp(P)
s = (m/A)**(1/n)
sx, sxx, sxxx = sp.diff(s, x), sp.diff(s, x, 2), sp.diff(s, x, 3)
Ax = sp.diff(A, x)
C = sp.binomial


def E(r, k):
    d = r - k
    if d == 1:
        return C(r, 2)*s**(r-2)*sx
    if d == 2:
        return C(r, 3)*s**(r-3)*sxx + 3*C(r, 4)*s**(r-4)*sx**2
    if d == 3:
        return (C(r, 4)*s**(r-4)*sxxx + 10*C(r, 5)*s**(r-5)*sx*sxx
                + 15*C(r, 6)*s**(r-6)*sx**3)
    raise ValueError(d)


def row(expr, monomials):
    """Coefficients of the given monomials in (c1,c2,c3), at x = 0."""
    e = sp.expand(sp.powsimp(sp.simplify(expr.subs(x, 0)), force=True))
    out = []
    for mono in monomials:
        term = sp.expand(e)
        for sym, k in mono:
            term = sp.diff(term, sym, k)/sp.factorial(k)
        out.append(sp.simplify(term.subs({c1: 0, c2: 0, c3: 0})))
    return out


MON2 = [[(c2, 1)], [(c1, 2)]]                       # p_x , p^2
MON3 = [[(c3, 1)], [(c1, 1), (c2, 1)], [(c1, 3)]]   # p_xx , p p_x , p^3


def main():
    ok = True

    contrib2 = {
        "reversed multiplier": s**(-(n-2))*m*(n-1)/s**2*(c2/2 + c1**2/n),
        "-eps A E_{n,n-2}":    -s**(-(n-2))*A*E(n, n-2),
        "order n-1 term":      -s**(-(n-2))*(n+1)/2*Ax*E(n-1, n-2),
    }
    claim2 = {
        "reversed multiplier": [(n-1)/2, (n-1)/n],
        "-eps A E_{n,n-2}":    [(n-1)*(n-2)/6, -(n-1)*(n-2)*(3*n-5)/(24*n)],
        "order n-1 term":      [0, (n-1)*(n-2)*(n+1)/(4*n)],
    }
    for k, v in contrib2.items():
        good = [sp.simplify(a-b) == 0 for a, b in zip(row(v, MON2), claim2[k])]
        ok = ok and all(good)
        print("order n-2 | %-22s %s" % (k, good))

    B2bg = A*((n**2-1)/6*sp.diff(P, x, 2)
              + (n**2-1)*(3*n+2)/(24*n)*sp.diff(P, x)**2)
    contrib3 = {
        "reversed multiplier": s**(-(n-3))*m*(n-1)*(n-2)/s**3
                               * (c3/6 + 7*c1*c2/(6*n) + c1**3/n**2),
        "-eps A E_{n,n-3}":    -s**(-(n-3))*A*E(n, n-3),
        "order n-1 term":      -s**(-(n-3))*(n+1)/2*Ax*E(n-1, n-3),
        "order n-2 coupling":  -s**(-(n-3))*B2bg*E(n-2, n-3),
    }
    claim3 = {
        "reversed multiplier": [(n-1)*(n-2)/6, 7*(n-1)*(n-2)/(6*n),
                                (n-1)*(n-2)/n**2],
        "-eps A E_{n,n-3}":    [(n-1)*(n-2)*(n-3)/24,
                                -(n-1)*(n-2)*(n-3)*(2*n-5)/(24*n),
                                (n-1)*(n-2)**2*(n-3)**2/(48*n**2)],
        "order n-1 term":      [0, (n-1)*(n-2)*(n-3)*(n+1)/(12*n),
                                -(n-1)*(n-2)*(n-3)*(n+1)*(3*n-8)/(48*n**2)],
        "order n-2 coupling":  [0, (n-1)*(n-2)*(n-3)*(n+1)/(12*n),
                                (n-1)*(n-2)*(n-3)*(n+1)*(3*n+2)/(48*n**2)],
    }
    for k, v in contrib3.items():
        good = [sp.simplify(a-b) == 0 for a, b in zip(row(v, MON3), claim3[k])]
        ok = ok and all(good)
        print("order n-3 | %-22s %s" % (k, good))

    print("every table entry verified:", ok)
    return ok


if __name__ == "__main__":
    main()

order n-2 | reversed multiplier    [True, True]
order n-2 | -eps A E_{n,n-2}       [True, True]
order n-2 | order n-1 term         [True, True]
order n-3 | reversed multiplier    [True, True, True]
order n-3 | -eps A E_{n,n-3}       [True, True, True]
order n-3 | order n-1 term         [True, True, True]
order n-3 | order n-2 coupling     [True, True, True]
every table entry verified: True
